In [1]:
import torch
import torch.nn as nn
import onnx
import os



symbols = [
    "1030", "1180","8300","8170","3030","3090","1050","4290"
    ]

# ── Model Definition ──────────────────────────────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2, output_size=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.attention    = nn.Linear(hidden_size * 2, 1)
        self.pos_bias     = nn.Parameter(torch.zeros(1))
        self.ln           = nn.LayerNorm(hidden_size * 2)
        self.dropout      = nn.Dropout(dropout)
        self.fc           = nn.Linear(hidden_size * 2, output_size)
        self.skip_fc      = nn.Linear(hidden_size * 2, output_size)
        self.output_blend = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        seq_len      = lstm_out.size(1)
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out_attn     = self.fc(self.dropout(self.ln(context)))
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


def infer_hyperparams(state_dict: dict) -> dict:
    """
    Infer model hyperparameters directly from state dict tensor shapes.

    Key derivations:
    lstm.weight_ih_l0  : shape (4*hidden, input_size)
        → hidden_size = shape[0] // 4
        → input_size  = shape[1]
    lstm.weight_ih_lN  : presence of layer index N determines num_layers
    fc.weight          : shape (output_size, hidden*2)
        → output_size = shape[0]  (1 for current 5d single-output models)
    """
    ih_l0       = state_dict["lstm.weight_ih_l0"]   # (4*H, input_size)
    hidden_size = ih_l0.shape[0] // 4
    input_size  = ih_l0.shape[1]

    num_layers = sum(
        1 for k in state_dict
        if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    output_size = state_dict["fc.weight"].shape[0]

    return {
        "input_size":  input_size,
        "hidden_size": hidden_size,
        "num_layers":  num_layers,
        "output_size": output_size,
    }


def convert(symbol: str):
    pt_path   = f"models/{symbol}/{symbol}_best_bilstm.pt"
    onnx_dir  = f"models/{symbol}"
    onnx_path = f"{onnx_dir}/{symbol}_bilstm.onnx"

    if not os.path.exists(pt_path):
        print(f"[SKIP] {symbol}: file not found → {pt_path}")
        return

    print(f"\n[{symbol}] Loading {pt_path} ...")

    state_dict = torch.load(pt_path, map_location="cpu", weights_only=True)

    if "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]
    elif "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    hp = infer_hyperparams(state_dict)
    print(f"[{symbol}] Inferred → input_size={hp['input_size']}, "
        f"hidden_size={hp['hidden_size']}, num_layers={hp['num_layers']}, "
        f"output_size={hp['output_size']}")

    model = StockPctChangeBiLSTMAttention(
        input_size  = hp["input_size"],
        hidden_size = hp["hidden_size"],
        num_layers  = hp["num_layers"],
        output_size = hp["output_size"],
    )
    model.load_state_dict(state_dict)
    model.eval()

    dummy_input = torch.randn(1, 1, hp["input_size"])

    os.makedirs(onnx_dir, exist_ok=True)

    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        opset_version=17,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input":  {0: "batch_size", 1: "seq_len"},
            "output": {0: "batch_size"},
        },
        do_constant_folding=True,
    )

    onnx.checker.check_model(onnx.load(onnx_path))
    print(f"[{symbol}] ✓ Saved & validated → {onnx_path}")


# ── Run all conversions ───────────────────────────────────────────────────────
if __name__ == "__main__":
    for sym in symbols:
        try:
            convert(sym)
        except Exception as e:
            print(f"[{sym}] ✗ Error: {e}")


[1030] Loading models/1030/1030_best_bilstm.pt ...
[1030] Inferred → input_size=42, hidden_size=64, num_layers=2, output_size=1


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


[1030] ✓ Saved & validated → models/1030/1030_bilstm.onnx

[1180] Loading models/1180/1180_best_bilstm.pt ...
[1180] Inferred → input_size=42, hidden_size=64, num_layers=2, output_size=1
[1180] ✓ Saved & validated → models/1180/1180_bilstm.onnx

[8300] Loading models/8300/8300_best_bilstm.pt ...
[8300] Inferred → input_size=42, hidden_size=64, num_layers=2, output_size=1
[8300] ✓ Saved & validated → models/8300/8300_bilstm.onnx

[8170] Loading models/8170/8170_best_bilstm.pt ...
[8170] Inferred → input_size=42, hidden_size=64, num_layers=3, output_size=1
[8170] ✓ Saved & validated → models/8170/8170_bilstm.onnx

[3030] Loading models/3030/3030_best_bilstm.pt ...
[3030] Inferred → input_size=42, hidden_size=128, num_layers=2, output_size=1
[3030] ✓ Saved & validated → models/3030/3030_bilstm.onnx

[3090] Loading models/3090/3090_best_bilstm.pt ...
[3090] Inferred → input_size=42, hidden_size=64, num_layers=2, output_size=1
[3090] ✓ Saved & validated → models/3090/3090_bilstm.onnx

[1050